# Day 4 - Transition into LLM Model Based + Fine Tuning Frontier Model

In [103]:
import os
import json
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets, Value
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from openai import OpenAI
from litellm import completion
from tqdm.notebook import tqdm
from IPython.display import display, Markdown
from pricer.evaluator import evaluate

In [57]:
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [58]:
openai = OpenAI(api_key=OPENAI_API_KEY)

In [59]:
ds = load_dataset("matthewyn/stocks")
train = ds["train"]
val = ds["validation"]
test = ds["test"]

README.md:   0%|          | 0.00/749 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.19M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/533k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/535k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1334 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1334 [00:00<?, ? examples/s]

In [5]:
display(Markdown(test[0]["Price History"]))

TREND: Bearish — the stock has been steadily declining with prices well below all longer-term MAs, indicative of a sustained downtrend.  
MOMENTUM: Momentum remains negative and is accelerating further into oversold territory, with RSI very low and trending upward from recent lows.  
MA_STRUCTURE: Price is consistently below the MA-50, MA-150, and MA-200, with these averages diverging further apart as the decline continues.  
VOLUME: Volume spiked notably during recent declines, suggesting high conviction behind the selling pressure.  
KEY_LEVELS: Support around 75, resistance near 170.  
RISK_FACTORS: The oversold RSI and positive short-term momentum shifts could signal a potential timing for a rebound, but overall trend remains downward.  
PREDICTION_CONTEXT: The dominant trend suggests continued downside risk over the next 30 days, with possible short-term oversold relief but no clear reversal signal.

In [135]:
print(f"Train size: {len(train)}, Val size: {len(val)}, Test size: {len(test)}")

Train size: 10668, Val size: 1334, Test size: 1334


In [61]:
def messages_for(item):
    message = f"Given this market summary where the last price was {item['Last Price']:.2f}, predict the closing price 30 days later. Return a single number; No explanation, no currency symbol. no extra text. Market summary:\n\n{item['Price History']}"
    return [{"role": "user", "content": message}]

In [62]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Given this market summary where the last price was 51.00, predict the closing price 30 days later. Return a single number; No explanation, no currency symbol. no extra text. Market summary:\n\nTREND: Bearish — the stock has been declining sharply, consistently below all moving averages, signaling downward momentum.  \nMOMENTUM: Momentum has been fading, with negative readings and RSI dropping into oversold territory, suggesting weakening price strength.  \nMA_STRUCTURE: Price remains below the MA-50, MA-150, and MA-200, with all MAs diverging downward, confirming a sustained downtrend.  \nVOLUME: Volume spikes on downturns indicate strong selling conviction, but recent decline in volume suggests less active capitulation.  \nKEY_LEVELS: Support around 51, resistance near 69.  \nRISK_FACTORS: The extremely oversold RSI and stabilization in recent volume could indicate a short-term bounce or reversal if buying interest resurges.  \nPREDICTION_CONTEXT: The o

In [63]:
def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [64]:
gpt_4__1_nano(test[0])

'45'

In [65]:
print(f"Actual price at t+30: {test[0]['Future Price']}")

Actual price at t+30: 50.0


In [134]:
evaluate(gpt_4__1_nano, train.select(range(1000)))

  0%|          | 0/200 [00:00<?, ?it/s]

80 99 147 148 143 19 42 21 57 1 36 185 144 86 118 243 163 140 143 178 91 8 73 3 40 10 145 282 339 270 337 390 487 487 516 459 454 413 500 514 439 269 346 323 322 239 172 193 241 160 143 187 185 205 198 109 174 148 160 226 172 122 88 92 119 87 83 125 30 59 49 159 117 195 184 207 422 334 398 480 461 415 366 347 307 273 258 249 179 182 205 200 234 228 190 153 177 79 11 35 25 32 81 203 236 183 236 280 284 264 245 274 262 205 249 209 193 157 174 155 109 132 141 162 210 194 211 178 75 33 84 110 184 53 84 144 128 79 127 100 176 166 230 356 338 338 299 381 370 395 426 522 450 483 414 429 458 549 505 441 408 250 153 186 95 28 34 4 40 52 48 6 102 225 204 148 86 163 212 310 245 276 312 355 333 467 475 407 321 234 204 156 173 80 148 46 117 61 6 50 

In [136]:
fine_tune_train = train.select(range(100))
fine_tune_validation = val.select(range(50))

In [137]:
print(f"The length of the fine-tuning training set is: {len(fine_tune_train)}")
print(f"The length of the fine-tuning validation set is: {len(fine_tune_validation)}")

The length of the fine-tuning training set is: 100
The length of the fine-tuning validation set is: 50


In [159]:
def messages_for(item):
    messages = f"Given this market summary where the last price was {item['Last Price']:.0f}, predict the closing price 30 days later. Return a single number. Market summary:\n\n{item['Price History']}"
    return [
        {"role": "user", "content": messages},
        {"role": "assistant", "content": f"{int(item['Future Price'])}"}
    ]

In [160]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Given this market summary where the last price was 1573, predict the closing price 30 days later. Return a single number. Market summary:\n\nTREND: Bearish — the stock has been declining sharply, with momentum and price below all key moving averages, indicating a dominant downtrend.  \nMOMENTUM: Momentum has been fading and remains negative with RSI in oversold territory, suggesting weakening bullish attempts and potential short-term stabilization.  \nMA_STRUCTURE: Price is below the 50, 150, and 200-day MAs, which are all diverging upwards but remain well above the current price, confirming downward pressure.  \nVOLUME: Volume spikes on recent declines indicate high conviction in continued selling; volume during rebounds is comparatively lower, further emphasizing bearish sentiment.  \nKEY_LEVELS: Support around 1550, Resistance near 1710, with key moving averages acting as dynamic resistance levels.  \nRISK_FACTORS: The persistent oversold RSI and vola

In [161]:
def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str + '}\n'
    return result.strip()

In [162]:
print(make_jsonl(fine_tune_train.select(range(5))))

{"messages": [{"role": "user", "content": "Given this market summary where the last price was 1573, predict the closing price 30 days later. Return a single number. Market summary:\n\nTREND: Bearish \u2014 the stock has been declining sharply, with momentum and price below all key moving averages, indicating a dominant downtrend.  \nMOMENTUM: Momentum has been fading and remains negative with RSI in oversold territory, suggesting weakening bullish attempts and potential short-term stabilization.  \nMA_STRUCTURE: Price is below the 50, 150, and 200-day MAs, which are all diverging upwards but remain well above the current price, confirming downward pressure.  \nVOLUME: Volume spikes on recent declines indicate high conviction in continued selling; volume during rebounds is comparatively lower, further emphasizing bearish sentiment.  \nKEY_LEVELS: Support around 1550, Resistance near 1710, with key moving averages acting as dynamic resistance levels.  \nRISK_FACTORS: The persistent overs

In [163]:
def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [164]:
write_jsonl(fine_tune_train, "jsonl/fine-tune/fine_tune_train.jsonl")

In [165]:
write_jsonl(fine_tune_validation, "jsonl/fine-tune/fine_tune_validation.jsonl")

In [166]:
with open("jsonl/fine-tune/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(
        file=f,
        purpose="fine-tune"
    )

In [167]:
train_file

FileObject(id='file-D3LP6CdLm62cDPCMigmvXv', bytes=125944, created_at=1777131546, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [168]:
with open("jsonl/fine-tune/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(
        file=f,
        purpose="fine-tune"
    )

In [169]:
validation_file

FileObject(id='file-UJXzQyotJrgGZ8jzn3WGGQ', bytes=65031, created_at=1777131547, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [170]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="stock_price_prediction"
)

FineTuningJob(id='ftjob-nt7Amo0LIhNT0hOJ6bS09J2M', created_at=1777131554, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-k5aZKTvbIC3mrQiSJVU7iUKf', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-D3LP6CdLm62cDPCMigmvXv', validation_file='file-UJXzQyotJrgGZ8jzn3WGGQ', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='stock_price_prediction', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=No

In [171]:
openai.fine_tuning.jobs.list(limit=5)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-nt7Amo0LIhNT0hOJ6bS09J2M', created_at=1777131554, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-nano-2025-04-14:personal:stock-price-prediction:DYZYF10m', finished_at=1777132183, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-k5aZKTvbIC3mrQiSJVU7iUKf', result_files=['file-JGgdBp7QDXfiZTHFRMwyzh'], seed=42, status='succeeded', trained_tokens=24513, training_file='file-D3LP6CdLm62cDPCMigmvXv', validation_file='file-UJXzQyotJrgGZ8jzn3WGGQ', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='stock_price_prediction', usage_metrics=None, shared_with_openai=False, eva

In [172]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [173]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-nt7Amo0LIhNT0hOJ6bS09J2M', created_at=1777131554, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-nano-2025-04-14:personal:stock-price-prediction:DYZYF10m', finished_at=1777132183, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-k5aZKTvbIC3mrQiSJVU7iUKf', result_files=['file-JGgdBp7QDXfiZTHFRMwyzh'], seed=42, status='succeeded', trained_tokens=24513, training_file='file-D3LP6CdLm62cDPCMigmvXv', validation_file='file-UJXzQyotJrgGZ8jzn3WGGQ', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='stock_price_prediction', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=N

In [174]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [175]:
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:personal:stock-price-prediction:DYZYF10m'

In [176]:
# The prompt

def test_messages_for(item):
    message = f"Given this market summary where the last price was {item['Last Price']:.2f}, predict the closing price 30 days later. Return a single number; No explanation, no currency symbol. no extra text. Market summary:\n\n{item['Price History']}"
    return [
        {"role": "user", "content": message},
    ]

In [177]:
test_messages_for(test[0])

[{'role': 'user',
  'content': 'Given this market summary where the last price was 51.00, predict the closing price 30 days later. Return a single number; No explanation, no currency symbol. no extra text. Market summary:\n\nTREND: Bearish — the stock has been declining sharply, consistently below all moving averages, signaling downward momentum.  \nMOMENTUM: Momentum has been fading, with negative readings and RSI dropping into oversold territory, suggesting weakening price strength.  \nMA_STRUCTURE: Price remains below the MA-50, MA-150, and MA-200, with all MAs diverging downward, confirming a sustained downtrend.  \nVOLUME: Volume spikes on downturns indicate strong selling conviction, but recent decline in volume suggests less active capitulation.  \nKEY_LEVELS: Support around 51, resistance near 69.  \nRISK_FACTORS: The extremely oversold RSI and stabilization in recent volume could indicate a short-term bounce or reversal if buying interest resurges.  \nPREDICTION_CONTEXT: The o

In [178]:
def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=10,
    )
    return response.choices[0].message.content

In [179]:
evaluate(gpt_4__1_nano_fine_tuned, train.select(range(1000)))

  0%|          | 0/200 [00:00<?, ?it/s]

151 34 694 144 405 72 86 283 286 256 206 141 50 10 903 355 353 60 21 119 274 143 55 213 237 241 261 128 109 258 227 135 374 91 699 393 393 174 355 446 364 140 223 86 195 48 159 33 486 155 0 87 256 129 27 370 178 54 32 251 88 284 183 23 97 207 156 46 46 51 54 49 61 115 152 52 146 334 50 466 404 312 15 164 4 210 136 14 327 189 147 180 92 49 303 102 116 75 65 30 84 82 198 225 226 415 242 260 383 298 246 311 434 270 312 245 214 192 189 130 281 178 159 99 210 225 316 172 570 156 288 94 146 78 49 93 218 203 57 100 252 164 245 408 339 367 360 419 400 478 458 670 539 512 439 371 468 453 482 370 456 236 67 531 292 142 432 102 285 151 21 70 110 128 277 367 91 357 260 349 101 171 241 536 555 614 283 589 415 462 303 1 68 41 7 169 373 210 202 140 

## Prepare data for Fine Tuning Open Source Model

In [124]:
full = concatenate_datasets([train, val, test])

In [125]:
print(f"Full dataset size: {len(full)}")

Full dataset size: 13336


In [126]:
full = full.map(lambda x: {"Price History": f"Given this market summary where the last price was {x['Last Price']:.0f}, predict the closing price 30 days later. Return a single number. Market summary:\n\n" + x["Price History"] + "\n\nPrice is Rp"})
full = full.map(lambda x: {"Future Price": f"{x['Future Price']:.0f}"})
full = full.cast_column("Future Price", Value("string"))

Map:   0%|          | 0/13336 [00:00<?, ? examples/s]

Map:   0%|          | 0/13336 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/13336 [00:00<?, ? examples/s]

In [127]:
print(f"Sample processed Price History:\n\n{full[0]['Price History']}")

Sample processed Price History:

Given this market summary where the last price was 1573, predict the closing price 30 days later. Return a single number. Market summary:

TREND: Bearish — the stock has been declining sharply, with momentum and price below all key moving averages, indicating a dominant downtrend.  
MOMENTUM: Momentum has been fading and remains negative with RSI in oversold territory, suggesting weakening bullish attempts and potential short-term stabilization.  
MA_STRUCTURE: Price is below the 50, 150, and 200-day MAs, which are all diverging upwards but remain well above the current price, confirming downward pressure.  
VOLUME: Volume spikes on recent declines indicate high conviction in continued selling; volume during rebounds is comparatively lower, further emphasizing bearish sentiment.  
KEY_LEVELS: Support around 1550, Resistance near 1710, with key moving averages acting as dynamic resistance levels.  
RISK_FACTORS: The persistent oversold RSI and volatility

In [128]:
full = full.remove_columns(["Last Price", "Start Date", "End Date", "Id", "Ticker"])

In [129]:
full = full.rename_columns({"Future Price": "completion", "Price History": "prompt"})

In [130]:
n = len(full)

In [131]:
completion_train = full.select(range(int(n * 0.8)))
completion_val = full.select(range(int(n * 0.8), int(n * 0.9)))
completion_test = full.select(range(int(n * 0.9), n))

In [132]:
print(f"Length of training set is: {len(completion_train)}, length of validation set is: {len(completion_val)}, length of test set is: {len(completion_test)}")

Length of training set is: 10668, length of validation set is: 1334, length of test set is: 1334


In [133]:
DatasetDict({
    "train": completion_train,
    "validation": completion_val,
    "test": completion_test
}).push_to_hub("matthewyn/stocks_prompt")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/matthewyn/stocks_prompt/commit/4b327c30cb65c0c2bf287068969119182c3c96f3', commit_message='Upload dataset', commit_description='', oid='4b327c30cb65c0c2bf287068969119182c3c96f3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/matthewyn/stocks_prompt', endpoint='https://huggingface.co', repo_type='dataset', repo_id='matthewyn/stocks_prompt'), pr_revision=None, pr_num=None)